Copyright 2026 Google LLC

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.

# 🚀 TPU High-Performance Computing Workshop
## Distributed FFT on Google Cloud TPUs

<table>
  <tr>
    <td style="padding-left: 24px;">
      ⏱ <strong>1 hour</strong> &nbsp;|&nbsp; 🎯 <strong>Intermediate</strong> &nbsp;|&nbsp; 🖥 <strong>TPU</strong>
    </td>
    <td>
      <a href="https://github.com/google-research/swirl-lm/blob/main/swirl_lm/ext/fft/dist_fft.py">
        <img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" alt="GitHub">
        View source on GitHub
      </a>
    </td>
  </tr>
</table>

**Welcome!** In this hands-on workshop, you'll learn how to leverage Google's Tensor Processing Units (TPUs) for high-performance scientific computing. We'll use a real-world **Distributed FFT** implementation as our teaching example — walking through JAX fundamentals, TPU mesh topology, and distributed computing patterns.

### What You'll Learn

By the end of this workshop, you will be able to:

1. ✅ **Initialize a JAX + TPU environment** — detect TPU devices and verify hardware topology
2. ✅ **Understand single-core TPU acceleration** — learn how a single-core TPU accelerate matrix operations and FFT
3. ✅ **Scale to multi-core distributed computation** — TPU mesh topology, data sharding, cross-device communication

---

### What is JAX?

**JAX** is a Python library from Google for high-performance numerical computing and it's designed to run on **accelerator hardware** like TPUs and GPUs.

| JAX Feature | What it does |
|---|---|
| **`jax.numpy`** | Multi-dimensional array operations (add, multiply, FFT, etc.) |
| **`shard_map`** | Runs the same function on all TPU cores simultaneously -- Single Program Multiple Data (SPMD) |
| **`lax.all_to_all` / `lax.ppermute`** | Cross-device data transfers |

---

### 🌊 What is Distributed FFT?

The **Fast Fourier Transform (FFT)** is one of the most important algorithms in scientific computing. It transforms signals between time/space and frequency domains and is used in:

- 🌤 **Weather simulation** — spectral methods for solving PDEs
- 🌊 **Fluid dynamics** — turbulence modeling in Navier-Stokes solvers
- 🧬 **Molecular dynamics** — Ewald summation for long-range interactions
- 📡 **Signal processing** — filtering, compression, spectral analysis

When datasets grow large or performance demands parallelization, a single device is no longer sufficient. A **distributed FFT** splits the computation across multiple TPU cores, coordinating data transfers over the high-bandwidth TPU interconnect to achieve both the memory capacity and speed required for large-scale scientific workloads.

In this workshop, we'll study an open-source implementation from the [Swirl-LM project](https://github.com/google-research/swirl-lm), a TPU-optimized computational fluid dynamics solver developed at Google Research.

> **📄 Reference Paper:**
> T. Lu, Y.-F. Chen, B. Hechtman, T. Wang, J. Anderson,
> *"Large-Scale Discrete Fourier Transform on TPUs"*,
> IEEE Access, vol. 9, pp. 93422–93432, 2021.
> [[IEEE](https://ieeexplore.ieee.org/document/9465154) | [arXiv](https://arxiv.org/abs/2002.03260)]

---

### 📏 How We Measure Correctness

Throughout this workshop, we compare TPU results against a NumPy ground truth using two relative error metrics:

- **Relative L2 error** — measures **overall quality**

 $$\frac{\sqrt{\sum_i |x_i - x_{\text{ref},i}|^2}}{\sqrt{\sum_i |x_{\text{ref},i}|^2}}$$

- **Relative L∞ error** — measures **peak error**

$$\frac{\max_i |x_i - x_{\text{ref},i}|}{\max_i |x_{\text{ref},i}|}$$

We report both because an algorithm can have small L2 but large L∞ (a few outlier elements), or vice versa.

<a id="section-1"></a>
## 🧩 Section 1: Initialize TPU Environment

### **⚙️ Setup steps:**
1. Go to **Runtime → Change runtime type → TPU** (select TPU hardware accelerator if available)
2. **Connect** to the selected TPU runtime. The notebook auto-detects available cores and adjusts accordingly
3. Run **Section 1 -- Initialize TPU Environment** to install dependencies and initialize TPU backend

**⚠️ Can't get a TPU?** Free-tier TPU access is "best effort" and may be unavailable during peak hours. If this happens, don't worry — **switch to CPU runtime** and the notebook will still run. You'll learn the same algorithm, code, and hardware acceleration concepts through the explanations and output — you'll just miss the hands-on experience of seeing TPU speedups firsthand.

**✅ Have [Colab Pro](https://colab.research.google.com/signup)?** You're all set! Colab Pro provides reliable TPU access, so you won't run into the availability issues mentioned above. With Colab Pro, you'll get access to **8 TPU cores**, enabling the full distributed multi-device experience — you'll see real cross-device communication (all-to-all, ppermute) in action.

**💡 Colab Pro vs Free Colab vs CPU Fallback:**
| | [Colab Pro](https://colab.research.google.com/signup) ($10/mo) | Free Colab (1 TPU core) | CPU Fallback (no TPU) |
|---|---|---|---|
| **Hardware** | Up to 8 TPU cores | 1 TPU core | CPU only |
| **Experience** | Full distributed multi-device FFT | Single-device FFT (distributed ops are no-ops) | All code runs; no TPU acceleration |
| **What you'll learn** | Full algorithm + real cross-device communication in action | Same algorithm + single-device TPU execution | Algorithm and concepts via explanations and output |

**This notebook works with all three!** All code runs correctly regardless of runtime. With 8 TPU cores, you'll get the full distributed experience.

In [ ]:
# @title 1.1 — Initialize TPU Environment { display-mode: "form" }

# ═══════════════════════════════════════════════════════════════════════════════
# This cell initializes the TPU environment.
# ═══════════════════════════════════════════════════════════════════════════════

# --- Step 1: Install required packages ---
# 'pip' is Python's package manager (like apt-get, brew, or conda).
# 'einops' is a library for concise array reshaping, used in the DistFFT code.
!pip install -q einops

# --- Step 2: Import libraries ---
# Python standard libraries
import collections           # Data structures (like C++ STL containers)
import functools             # Function utilities (partial application, etc.)
import time                  # Wall-clock timing
from typing import Text, Tuple  # Type hints (for documentation, not enforced)

# JAX — the main library for TPU computation
import jax                   # Core JAX: device management, JIT compilation
from jax import lax          # Low-level primitives (all_to_all, ppermute, etc.)
from jax.experimental import mesh_utils  # Helpers for creating device meshes

# shard_map — for writing SPMD (Single Program, Multiple Data) code
# (Its location moved between JAX versions, so we try both)
try:
    from jax.shard_map import shard_map
except ModuleNotFoundError:
    from jax.experimental.shard_map import shard_map

import jax.numpy as jnp      # JAX's array library (like NumPy, but runs on TPU)
from jax.sharding import Mesh                # Logical device mesh
from jax.sharding import PartitionSpec as P  # How to distribute arrays across devices

# NumPy — Python's standard CPU array library (used as ground truth / reference)
import numpy as np

# --- Step 3: Detect hardware ---
num_devices = len(jax.devices())  # How many TPU cores (or CPUs) are available?
backend = jax.default_backend()   # 'tpu', 'gpu', or 'cpu'

if backend == 'tpu':
    if num_devices >= 8:
        print(f"✅ TPU ready with {num_devices} devices — full distributed mode!")
    elif num_devices > 1:
        print(f"✅ TPU ready with {num_devices} devices — partial distributed mode.")
    else:
        print(f"⚠️ Only 1 TPU device detected (free Colab tier).")
        print(f"   The notebook will run in single-device mode (partition 1,1,1).")
        print(f"   For multi-core access, consider Colab Pro:")
        print(f"   → https://colab.research.google.com/signup")
else:
    print(f"🟡 Running on {backend.upper()} (TPU not available).")
    print(f"   No problem — the entire notebook runs on {backend.upper()} too!")
    print(f"   You'll learn the same algorithm and code structure.")
    if backend == 'cpu':
        print(f"   Execution will be slower than TPU, but all sections work correctly.")
    print(f"\n   💡 To try TPU: Runtime → Change runtime type → TPU")

print(f"Available {backend.upper()} devices: {num_devices}")
for i, d in enumerate(jax.devices()):
    print(f"   Device {i}: {d.platform.upper()} — {d.device_kind}")


# ═══════════════════════════════════════════════════════════════════════════════
# Global configuration
# ═══════════════════════════════════════════════════════════════════════════════

NUM_TRIALS = 10

# Map core count → balanced 3D partition
PARTITION_MAP = {1: (1, 1, 1), 2: (2, 1, 1), 4: (2, 2, 1), 8: (2, 2, 2)}
# Find the largest supported partition ≤ NUM_CORES
supported = sorted([k for k in PARTITION_MAP if k <= num_devices], reverse=True)
num_cores = supported[0]
partition = PARTITION_MAP[num_cores]

M = 256  # Each dimension: 256 — total grid is 256³ ≈ 16M elements
GLOBAL_SHAPE = (M, M, M)


# ═══════════════════════════════════════════════════════════════════════════════
# Helper functions for error metrics
# ═══════════════════════════════════════════════════════════════════════════════

def relative_l2_error(result, expected):
    """Relative L2 (Frobenius) error: ||result - expected||_2 / ||expected||_2.

    Measures the overall quality of the result across all elements.
    A small L2 error means the result is accurate on the whole.

    Args:
      result: The computed result array (any dtype, will be upcast to
              complex128 for accurate comparison).
      expected: The ground-truth / reference array.

    Returns:
      A scalar: the relative L2 error.
    """
    result = np.array(result, dtype=np.complex128)
    expected = np.array(expected, dtype=np.complex128)
    return np.linalg.norm(result - expected) / np.linalg.norm(expected)

def relative_linfinite_error(result, expected):
    """Relative L-infinity (max) error: ||result - expected||_inf / ||expected||_inf.

    Measures the worst-case element error, normalized by the peak signal
    magnitude. A small L-inf error means no individual element deviates
    significantly from the reference.

    Args:
      result: The computed result array (any dtype, will be upcast to
              complex128 for accurate comparison).
      expected: The ground-truth / reference array.

    Returns:
      A scalar: the relative L-infinity error.
    """
    result = np.array(result, dtype=np.complex128)
    expected = np.array(expected, dtype=np.complex128)
    return np.max(np.abs(result - expected)) / np.max(np.abs(expected))

/bin/sh: line 1: pip: command not found
✅ TPU ready with 8 devices — full distributed mode!
Available TPU devices: 8
   Device 0: TPU — TPU v3
   Device 1: TPU — TPU v3
   Device 2: TPU — TPU v3
   Device 3: TPU — TPU v3
   Device 4: TPU — TPU v3
   Device 5: TPU — TPU v3
   Device 6: TPU — TPU v3
   Device 7: TPU — TPU v3


<a id="section-2"></a>
## 🔧 Section 2: Single-Core TPU Acceleration

Before we distribute work across multiple cores, let's understand what makes a **single TPU core** so powerful for scientific computing.

### TPU Hardware: MXU and VPU

Each TPU core contains two specialized processing units:

| Unit | Full Name | What it does | Key operations |
|------|-----------|-------------|----------------|
| **MXU** | Matrix Multiply Unit | A 128×128 systolic array optimized for matrix operations | `matmul`, `einsum` — the heavy-lifting engine |
| **VPU** | Vector Processing Unit | Handles element-wise and vector operations | `add`, `multiply`, `sin`, `cos`, `exp` — the "glue" between matmuls |

```
  ┌─────────────────────────────────────────────────────────────────────────┐
  │                                 TPU Core                                │
  │                                                                         │
  │  ┌────────────────────────────────┐  ┌───────────────────────────────┐  │
  │  │   MXU 128x128 Systolic Array   │  │  VPU Element-wise Operations  │  │
  │  │   (matmul, einsum)             │  │  (add, mul, sin, cos, exp)    │  │
  │  └────────────────────────────────┘  └───────────────────────────────┘  │
  │                                     ↕                                   │
  │                    ┌─────────────────────────────────┐                  │
  │                    │   HBM (High Bandwidth Memory)   │                  │
  │                    └─────────────────────────────────┘                  │
  └─────────────────────────────────────────────────────────────────────────┘
```

> **💡 Why this matters for FFT:** The distributed FFT implementation uses `jnp.einsum` for data shuffling (runs on MXU), `jnp.fft.fft` for local FFT transforms (runs on MXU), and `sin`/`cos` for twiddle factors (runs on VPU). Understanding which operations run on which unit helps explain why TPU is so effective for this workload.

The cell below demonstrates the MXU in action — we'll compare matrix multiplication on CPU (NumPy) vs TPU (JAX). On TPU, the MXU delivers massive speedups for this operation.

In [ ]:
# @title 2.1 — MXU in Action: Matrix Multiply on TPU vs CPU { display-mode: "form" }

# ═══════════════════════════════════════════════════════════════════════════════
# Prerequisite Check: Ensure cell 1.1 has been run
# ═══════════════════════════════════════════════════════════════════════════════
_missing_vars = [v for v in ['jax', 'jnp', 'np', 'NUM_TRIALS']
                 if v not in globals()]
if _missing_vars:
    raise RuntimeError(
        "⚠️  Please run cell \"1.1 — Initialize TPU Environment\" first!\n\n"
        "   Cell 2.1 depends on libraries and variables initialized in cell 1.1:\n"
        f"   Missing: {', '.join(_missing_vars)}\n\n"
        "   👉 Scroll up to Section 1 and run cell 1.1, then re-run this cell."
    )

# ═══════════════════════════════════════════════════════════════════════════════
# This cell demonstrates the TPU's MXU (Matrix Multiply Unit) in action.
# We compare matrix multiplication on CPU (NumPy) vs TPU (JAX) to show
# how the MXU's 128×128 systolic array accelerates matrix operations.
# ═══════════════════════════════════════════════════════════════════════════════

# --- Step 1: Create a 2D array (matrix) ---
# np.random.randn(2048, 2048) creates a matrix filled with random numbers.
# .astype(np.float32) sets the data type to 32-bit floating point.
x_np = np.random.randn(2048, 2048).astype(np.float32)

# Transfer the same data to JAX (which places it on the TPU).
# Think of this as copying data from host memory to device memory.
x_jax = jnp.array(x_np)
print(f"📝 Experiment setup: Create a 2048 x 2048 matrix with random numbers")
print(f"     NumPy array:  shape={x_np.shape}, data type={x_np.dtype}")
print(f"                   allocated in the CPU's host RAM")
print(f"     JAX array:    shape={x_jax.shape}, data type={x_jax.dtype}")
print(f"                   transferred to the device HBM on {x_jax.devices()}")

# Warmup: first JAX call may be slower due to initial setup.
# .block_until_ready() waits for the TPU computation to finish
# (JAX operations are asynchronous by default).
_ = jnp.matmul(x_jax, x_jax).block_until_ready()

# --- Step 2: Benchmark CPU vs TPU ---
# Time NumPy (runs on CPU)
start = time.perf_counter()
for _ in range(NUM_TRIALS):
    result_cpu = np.matmul(x_np, x_np)
numpy_matmul_time = (time.perf_counter() - start) / NUM_TRIALS

# Time JAX (runs on TPU if available)
start = time.perf_counter()
for _ in range(NUM_TRIALS):
    result_tpu = jnp.matmul(x_jax, x_jax).block_until_ready()
jax_matmul_time = (time.perf_counter() - start) / NUM_TRIALS

backend = jax.default_backend()
print(f"\n⏱  Measure the time to multiply two (2048 x 2048) matrices together:")
print(f"     NumPy (CPU):         {numpy_matmul_time*1000:.1f} ms")
print(f"     JAX ({backend.upper()}):    {jax_matmul_time*1000:>10.1f} ms")

if backend == 'tpu':
    print(f"     Speedup:             {numpy_matmul_time/jax_matmul_time:.1f}x 🚀")
    print(f"\n💡 The TPU's Matrix Multiply Unit (MXU) makes this fast!")
    print(f"   The MXU is a 128×128 systolic array that can perform")
    print(f"   thousands of multiply-accumulate operations per cycle.")
else:
    print(f"\n 📝 On {backend.upper()}, JAX and NumPy use the same hardware, so their")
    print(f"    computational times are similar. On TPU, the Matrix Multiply")
    print(f"    Unit (MXU) delivers massive speedups — often 10-100x!")

# --- Step 3: Correctness Check — CPU vs TPU results ---
matmul_rel_l2 = relative_l2_error(result_tpu, result_cpu)
matmul_rel_linf = relative_linfinite_error(result_tpu, result_cpu)

print(f"\n✅ Correctness Check:")
print(f"     Relative L2 error:    {matmul_rel_l2:.6e}")
print(f"     Relative L∞ error:    {matmul_rel_linf:.6e}")

📝 Experiment setup: Create a 2048 x 2048 matrix with random numbers
     NumPy array:  shape=(2048, 2048), data type=float32
                   allocated in the CPU's host RAM
     JAX array:    shape=(2048, 2048), data type=float32
                   transferred to the device HBM on {TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0)}

⏱  Measure the time to multiply two (2048 x 2048) matrices together:
     NumPy (CPU):         374.8 ms
     JAX (TPU):           2.0 ms
     Speedup:             187.3x 🚀

💡 The TPU's Matrix Multiply Unit (MXU) makes this fast!
   The MXU is a 128×128 systolic array that can perform
   thousands of multiply-accumulate operations per cycle.

✅ Correctness Check:
     Relative L2 error:    3.537945e-07
     Relative L∞ error:    4.899409e-07


### 🌊 FFT on TPU vs CPU

Now let's see how the TPU handles **Fast Fourier Transforms (FFT)** — the core operation in this workshop's distributed algorithm, which involves:
- **Butterfly operations** — structured sparse matrix multiplies at each stage
- **Twiddle factors** — element-wise complex multiplications (`sin`/`cos` on the VPU)

JAX's `jnp.fft.fft` is compiled via XLA and can leverage both the **MXU** (for the butterfly matrix operations) and the **VPU** (for twiddle factors), making it significantly faster than NumPy's CPU-based FFT.




> **💡 Why this matters:** The distributed FFT we build later calls `jnp.fft.fft` on each TPU core for the *local* portion of the FFT transform. The speedup and error bounds you see below directly translate to per-core behavior in the distributed algorithm.

#### 📊 What We Benchmark

Rather than benchmarking FFT alone, we compute the full **FFT → kernel multiply → iFFT** pipeline:

```
input ──▶ FFT (forward) ──▶ × kernel ──▶ iFFT (inverse) ──▶ output
```

This is the standard pattern in scientific computing — transform to frequency domain, apply a spectral operation (e.g., filtering), then transform back. It's exactly the pipeline our distributed FFT will use later.

Proper benchmarking on TPU requires some care:

1. **Warmup** — The first call may be slower due to initial setup. Warmup iterations are excluded from timing.
2. **`block_until_ready()`** — JAX operations are **asynchronous**. We must wait for the TPU to finish before reading the timer.
3. **Multiple iterations** — Report mean for statistical reliability, just as you would for any HPC benchmark.

In [ ]:
# @title 2.2 — Single-Core FFT on TPU vs. NumPy FFT on CPU { display-mode: "form" }

# ═══════════════════════════════════════════════════════════════════════════════
# Prerequisite Check: Ensure cell 1.1 has been run
# ═══════════════════════════════════════════════════════════════════════════════
_missing_vars = [v for v in ['jax', 'jnp', 'np', 'NUM_TRIALS', 'partition',
                             'GLOBAL_SHAPE', 'M', 'relative_l2_error',
                             'relative_linfinite_error']
                 if v not in globals()]
if _missing_vars:
    raise RuntimeError(
        "⚠️  Please run cell \"1.1 — Initialize TPU Environment\" first!\n\n"
        "   Cell 2.2 depends on libraries and variables initialized in cell 1.1:\n"
        f"   Missing: {', '.join(_missing_vars)}\n\n"
        "   👉 Scroll up to Section 1 and run cell 1.1, then re-run this cell."
    )

# ═══════════════════════════════════════════════════════════════════════════════
# This cell compares the Fast Fourier Transform (FFT) on CPU (NumPy) vs
# TPU (JAX). FFT is the core operation in our distributed FFT algorithm —
# each TPU core runs a local FFT on its shard of data.
# ═══════════════════════════════════════════════════════════════════════════════

# ═══════════════════════════════════════════════════════════════════════════════
# Helper Functions: Data Generation & Validation
# ═══════════════════════════════════════════════════════════════════════════════
np.random.seed(123456)

def gen_random(local_shape, core_index, seeds):
    """Generate random complex data for a given TPU core.

    Each core gets unique random data based on its coordinates,
    ensuring reproducible results across runs.
    """
    key_0 = jax.random.key(
        seeds[0] + core_index[0] + 10 * core_index[1] + 100 * core_index[2])
    key_1 = jax.random.key(
        seeds[1] + core_index[0] + 10 * core_index[1] + 100 * core_index[2])
    a = jax.random.normal(key_0, local_shape, dtype=jnp.float32)
    b = jax.random.normal(key_1, local_shape, dtype=jnp.float32)
    return jnp.complex64(a + jnp.complex64(1j) * b)


def make_input_fn(seeds=(10061, 23399)):
    """Create an input generator for the FFT benchmark."""
    def input_fn(local_shape, core_index):
        return gen_random(local_shape, core_index, seeds)
    return input_fn


def make_kernel_fn(seeds=(12143, 16573)):
    """Create a spectral kernel generator (normalized)."""
    def kernel_fn(local_shape, core_index):
        kernel_local = gen_random(local_shape, core_index, seeds)
        kernel_norm = jax.numpy.linalg.norm(kernel_local)
        return kernel_local / kernel_norm
    return kernel_fn


def build_global_array(fn, global_shape, partition):
    """Reconstruct a full global array from per-core generation function.

    Calls fn(local_shape, core_index) for every core and stitches
    the local shards into one global array. Output dtype is complex64.
    """
    px, py, pz = partition
    gx, gy, gz = global_shape
    nx, ny, nz = gx // px, gy // py, gz // pz
    global_arr = np.zeros(global_shape, dtype=np.complex64)
    for ix in range(px):
        for iy in range(py):
            for iz in range(pz):
                local = np.array(
                    fn((nx, ny, nz), (ix, iy, iz)), dtype=np.complex64)
                global_arr[
                    ix * nx : (ix + 1) * nx,
                    iy * ny : (iy + 1) * ny,
                    iz * nz : (iz + 1) * nz,
                ] = local
    return global_arr

def compute_numpy_result(global_input, global_kernel):
    """Compute expected result using numpy.fft (ground truth).
    """
    input_f64 = global_input.astype(np.complex128)
    kernel_f64 = global_kernel.astype(np.complex128)
    fft_input = np.fft.fftn(input_f64)
    result = np.fft.ifftn(fft_input * kernel_f64)
    return result.astype(np.complex128)

def compute_jax_result(global_input, global_kernel):
    """Compute jax result using jax.fft.
    """
    fft_input = jnp.fft.fftn(global_input)
    expected = jnp.fft.ifftn(fft_input * global_kernel)
    return expected

# ═══════════════════════════════════════════════════════════════════════════════
# Step 1: Generate input data
# ═══════════════════════════════════════════════════════════════════════════════

# Create input and kernel generators
input_fn = make_input_fn()
kernel_fn = make_kernel_fn()

# Build global arrays in float32 (complex64)
numpy_input = build_global_array(input_fn, GLOBAL_SHAPE, partition)
numpy_kernel = build_global_array(kernel_fn, GLOBAL_SHAPE, partition)

# Create JAX arrays (already complex64, no conversion needed)
jax_input = jnp.array(numpy_input)
jax_kernel = jnp.array(numpy_kernel)

print(f"\n📝 Experiment setup: Prepare a 3D signal with {M}×{M}×{M} = {M**3:,} random elements")
print(f"     Input data:   shape={numpy_input.shape}, dtype={numpy_input.dtype}")
print(f"     JAX array:    shape={jax_input.shape}, dtype={jax_input.dtype}")
print(f"                   transferred to device HBM on {jax_input.devices()}")

# ═══════════════════════════════════════════════════════════════════════════════
# Step 2: Benchmark CPU vs TPU (Single-Core FFT)
# ═══════════════════════════════════════════════════════════════════════════════

# Warmup
_ = compute_jax_result(jax_input, jax_kernel).block_until_ready()

# Time NumPy 3D FFT (runs on CPU)
start = time.perf_counter()
for _ in range(NUM_TRIALS):
    numpy_result = compute_numpy_result(numpy_input, numpy_kernel)
numpy_time = (time.perf_counter() - start) / NUM_TRIALS

# Time JAX 3D FFT (runs on TPU in float32)
start = time.perf_counter()
for _ in range(NUM_TRIALS):
    jax_result = compute_jax_result(jax_input, jax_kernel).block_until_ready()
jax_time = (time.perf_counter() - start) / NUM_TRIALS

print(f"\n⏱  3D FFT -> kernel multiply -> 3D iFFT ({M}×{M}×{M} = {M**3:,} elements):")
print(f"     NumPy (CPU):        {numpy_time*1000:.2f} ms")
print(f"     JAX ({backend.upper()}):     {jax_time*1000:>10.2f} ms")

if numpy_time > jax_time:
    print(f"     Speedup:            {numpy_time/jax_time:.1f}x 🚀")

# --- Verify correctness ---
# Compare JAX float32 result against NumPy float64 ground truth using
# both L2 (overall quality) and L∞ (peak error) relative metrics.
jax_fft_rel_l2 = relative_l2_error(jax_result, numpy_result)
jax_fft_rel_linf = relative_linfinite_error(jax_result, numpy_result)

print(f"\n✅ Correctness check (JAX vs NumPy ground truth):")
print(f"     Relative L2 error:    {jax_fft_rel_l2:.2e}   (overall quality)")
print(f"     Relative L∞ error:    {jax_fft_rel_linf:.2e}   (peak error)")

if backend == 'tpu':
    print(f"\n💡 JAX compiles FFT via XLA, leveraging the TPU's MXU for butterfly")
    print(f"   operations and VPU for twiddle factors. This is why the local FFT")
    print(f"   in our distributed algorithm runs so fast on each TPU core!")
else:
    print(f"\n📝 On {backend.upper()}, JAX and NumPy use similar CPU libraries, so")
    print(f"   performance is comparable. On TPU, XLA-compiled FFT delivers")
    print(f"   significant speedups — the foundation of our distributed algorithm.")


📝 Experiment setup: Prepare a 3D signal with 256×256×256 = 16,777,216 random elements
     Input data:   shape=(256, 256, 256), dtype=complex64
     JAX array:    shape=(256, 256, 256), dtype=complex64
                   transferred to device HBM on {TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0)}

⏱  3D FFT -> kernel multiply -> 3D iFFT (256×256×256 = 16,777,216 elements):
     NumPy (CPU):        1215.56 ms
     JAX (TPU):          34.89 ms
     Speedup:            34.8x 🚀

✅ Correctness check (JAX vs NumPy ground truth):
     Relative L2 error:    3.87e-07   (overall quality)
     Relative L∞ error:    3.99e-07   (peak error)

💡 JAX compiles FFT via XLA, leveraging the TPU's MXU for butterfly
   operations and VPU for twiddle factors. This is why the local FFT
   in our distributed algorithm runs so fast on each TPU core!


<a id="section-3"></a>
## 🌐 Section 3: Distributed Communication on Multi-Core TPU

Now that we understand how a single TPU core accelerates computation with its MXU and VPU, let's scale to **multiple cores**. TPUs are connected in a 2D or 3D torus topology — each chip is connected to its neighbors, forming a mesh (similar to how nodes are connected in an HPC cluster). JAX lets us map this physical topology into a **logical mesh** of named axes.

### 📦 How Data is Distributed

When we have a global **3D array** of shape `(nx, ny, nz)` and a **3D mesh** of TPU cores with the partition `(cx, cy, cz)`, each TPU core holds a **local shard** of shape:

$$\text{local shape} = \left(\frac{n_x}{c_x},\; \frac{n_y}{c_y},\; \frac{n_z}{c_z}\right)$$

This is standard **domain decomposition**. Each core "owns" a sub-block of the global grid. The full array is reconstructed by tiling the shards. For example, with a `512³` global array and `(2, 2, 2)` partition, each of the **8 TPU cores** holds a `256 × 256 × 256` shard.

---

### 🧱 What is `jax.lax`?

`jax.lax` is JAX's **lower-level API** — think of it as the "assembly language" of JAX. While `jax.numpy` provides familiar high-level array operations (like matrix multiply, FFT, etc.), `jax.lax` gives you access to **hardware-level primitives**, especially for **cross-device communication** required in distributed computing.

| `jax.lax` primitive | What it does |
|---|---|
| `lax.all_to_all` | Redistribute data across a mesh axis so every core sends a chunk to every other |
| `lax.ppermute` | Send data along a specific (source, target) mapping for a direct point-to-point transfer |
| `lax.axis_index` | Get the current device's index in the mesh |
| `lax.while_loop` | A loop that can be compiled to efficient TPU machine code |

These are the building blocks for **distributed communication** on TPU. The distributed FFT uses all four.

---

### 🔗 Key Concept: `shard_map`

`shard_map` is JAX's way of writing **Single Program, Multiple Data (SPMD)** code. You write **one function**, and it runs on **all devices simultaneously**. Each device operates on its own local shard of the data.

In our distributed FFT codebase, `shard_map` is invoked via `functools.partial` as a **decorator**:

```python
# Pattern used in dist_fft.py and this workshop:
#
#   functools.partial(shard_map, ...) creates a pre-configured decorator
#   that wraps any function into an SPMD program.
#
#   mesh      = which devices to use and how they're organized
#   in_specs  = how to partition input arrays across the mesh
#   out_specs = how to partition output arrays across the mesh
#   check_rep = False allows non-replicated outputs

@functools.partial(
    shard_map, mesh=mesh,
    in_specs=P('x', 'y', 'z', None, None, None),
    out_specs=P('x', 'y', 'z', None, None, None),
    check_rep=False)
def do_transform(partitioned_input):
    # This runs on EACH device independently
    x_axis_index = lax.axis_index('x')  # Which core am I?
    # ... do local compute + cross-device communication ...
    return result
```

> **📝 Note:** Inside `shard_map`, you can call `lax.all_to_all` and `lax.ppermute` to communicate with other devices. These are the ways to exchange data between cores.

---

### 📐 The Distributed FFT Algorithm

Now we get to the heart of the distributed computing story — understanding how to compute a **global FFT** across multiple TPU cores using the communication primitives we just learned.

#### 🔍 The Cooley-Tukey Factorization

The standard Discrete Fourier Transform (DFT) of a signal $x[n]$ of length $N$ is:

$$X[k] = \sum_{n=0}^{N-1} x[n] \, W_N^{nk}, \qquad W_N = e^{-2\pi j / N}$$

The key idea: factorize $N = P \times L$ where:
- $P$ = number of TPU cores (partition size along one axis)
- $L$ = local elements per core ($L = N / P$)

Using the index mapping $n = Pl + p$ and $k = Lq + m$, the classic **Cooley-Tukey decomposition** factors the DFT into:

$$X[Lq + m] = \sum_{p=0}^{P-1} \underbrace{W_P^{pq}}_{\text{Ring DFT}} \cdot \underbrace{W_N^{pm}}_{\text{Twiddle}} \cdot \underbrace{\left[\sum_{l=0}^{L-1} x[Pl + p] \cdot W_L^{lm}\right]}_{\text{Local FFT (size } L \text{)}}$$

This gives us a **4-step distributed FFT pipeline**:

#### 🔄 The Distributed FFT Pipeline

The distributed FFT along **one axis** has 4 steps (as below). Steps 1 and 4 involve **inter-device communication**, while steps 2 and 3 are purely **local** computation on each core:

| Step | Operation | Cross-Core Communication? | What it does |
|:----:|-----------|:--------------:|-------------|
| **1** | **Shuffle + All-to-All** | ✅ `all_to_all` | Redistribute data so each core holds strided samples |
| **2** | **Local FFT** | ❌ local | Each core runs a size-$L$ FFT on its local data |
| **3** | **Twiddle Multiply** | ❌ local | Multiply by phase correction $W_N^{pm}$ |
| **4** | **Ring DFT** | ✅ `ppermute` | $P$-point DFT across cores via ring communication |

```
Input ──▶ [Shuffle + All-to-All] ──▶ [Local FFT] ──▶ [Twiddle × Wₙ] ──▶ [Ring DFT] ──▶ Output
              (cross-device)        (per-device)      (per-device)    (cross-device)
```

For a **3D FFT**, this pipeline runs sequentially along each axis: **axis-0 → axis-1 → axis-2**.

#### 🏗️ Full DistFFT Implementation

Now let's load the complete [`DistFFT` class](https://github.com/google-research/swirl-lm/blob/main/swirl_lm/ext/fft/dist_fft.py) from the open-source [Swirl-LM](https://github.com/google-research/swirl-lm) project. This implements the 4-step pipeline we just discussed, using the `all_to_all` and `ppermute` communication patterns from above.

**📝 Note:** The cell below contains the full implementation. You don't need to read every line; the key methods are:

 | Method | Purpose |
 |--------|---------|
 | `__init__` | Constructor — sets up axis names, partition shape, shuffle equations, ring permutations |
 | `_get_shuffle_op` | Builds the permutation matrix for data redistribution (Step 1) |
 | `_get_gather_adjust_op` | Build post-all-to-all data gathering operator (Step 1) |
 | `_get_correction_factor` | Computes twiddle factors $W_N^{pm}$ (Step 3) |
 | `partitioned_fft_1d` | **The core algorithm** — runs the 4-step pipeline on one axis |
 | `fft_3d_perf` | Full 3D forward FFT → kernel multiply → inverse FFT pipeline |

 ---

### ▶️ Running & Validating the Distributed FFT

Let's put it all together! We'll:
1. Create an instance of the `DistFFT` class (this sets up the mesh and communication patterns)
2. Generate random input data and a spectral kernel
3. Run the full 3D FFT pipeline on TPU: **forward FFT → kernel multiply → inverse FFT**
4. Validate the result against a known-correct reference implementation (NumPy's single-machine FFT running on CPU)

> **📝 Why validate?** In any distributed algorithm, correctness verification is critical. We compute the same operation two ways — once with our distributed TPU code, once with a trusted single-machine library — and compare the results. If the maximum error is within floating-point tolerance, we know our distributed code is correct.

In [ ]:
# @title 3.1 — Distributed FFT with Cross-Core Communication { display-mode: "form" }

# ═══════════════════════════════════════════════════════════════════════════════
# Prerequisite Check: Ensure cell 2.2 has been run
# ═══════════════════════════════════════════════════════════════════════════════
_missing_vars = [v for v in ['input_fn', 'kernel_fn', 'numpy_result',
                             'numpy_time', 'jax_time', 'numpy_input',
                             'numpy_kernel']
                 if v not in globals()]
if _missing_vars:
    raise RuntimeError(
        "⚠️  Please run cell \"2.2 — Single-Core FFT on TPU vs. NumPy FFT on CPU\" first!\n\n"
        "   Cell 3.1 depends on variables defined in cell 2.2:\n"
        f"   Missing: {', '.join(_missing_vars)}\n\n"
        "   👉 Scroll up to Section 2 and run cell 2.2, then re-run this cell."
    )

# Source: https://github.com/google-research/swirl-lm/blob/main/swirl_lm/ext/fft/dist_fft.py

SUPPORTED_BACKENDS = ['tpu', 'gpu', 'cpu']

class DistFFT():
  """Distributed FFT — M-way split Cooley-Tukey on TPU.

  Open source implementation from the Swirl-LM project:
  https://github.com/google-research/swirl-lm

  Usage:
      transformer = DistFFT(['x', 'y', 'z'], [2, 2, 2])
      dist_fn = transformer.make_fft_3d_fn(shape, input_fn, kernel_fn)
      result = dist_fn()
  """

  def __init__(
      self,
      axis_names: Tuple[Text, Text, Text],
      partition: Tuple[int, int, int],
      backend: str = 'tpu',
  ) -> None:
    """Initialize the distributed FFT handler.

    Args:
      axis_names: Names of the three mesh axes, e.g. ('x', 'y', 'z').
      partition: The 3D partition shape, e.g. (2, 2, 2) for 8 cores.
      backend: One of 'tpu', 'gpu', 'cpu'.
    """
    if backend not in SUPPORTED_BACKENDS:
      raise ValueError('Unsupported backend type %s. Supported types are %s' %
                       (backend, SUPPORTED_BACKENDS))
    self._backend = backend
    self._axis_names = axis_names
    self._partition = partition
    # Einsum equations for the shuffle operation along each axis
    self._shuffle_eqs = ['ijk,il->ljk', 'ijk,jl->ilk', 'ijk,kl->ijl']
    # Transpose permutations to move the FFT axis to the last position
    self._fft_perms = [[2, 1, 0], [0, 2, 1], [0, 1, 2]]
    # Ring communication: each core sends to (i+1) % P
    self._source_target_pairs = [
        [(i, (i + 1) % l) for i in range(l)] for l in self._partition
    ]
    self._devices = None
    self._mesh = None

  # ─── Mesh Setup ──────────────────────────────────────────────────────────

  def _create_mesh(self):
    """Create a 3D device mesh (lazy initialization)."""
    if self._mesh is not None:
      return
    num_devices_needed = np.prod(self._partition)
    devices = jax.devices()[:num_devices_needed]
    self._devices = mesh_utils.create_device_mesh(
        (self._partition[0], self._partition[1], self._partition[2]),
        devices=devices,
        allow_split_physical_axes=True,
    )
    self._mesh = Mesh(self._devices, axis_names=(
        self._axis_names[0], self._axis_names[1], self._axis_names[2]))

  # ─── Building Blocks ────────────────────────────────────────────────────

  def _exp_itheta(self, theta: jax.Array) -> jax.Array:
    """Compute exp(i·θ) = cos(θ) + i·sin(θ)."""
    return jnp.cos(theta) + jnp.complex64(1.0j) * jnp.sin(theta)

  def _get_shuffle_op(self, n, m, l, axis, margin_width=0):
    """Build the einsum-based permutation matrix for data shuffling.

    Creates a one-hot matrix that rearranges elements cyclically so that
    every m-th element ends up in the same segment — preparing for all_to_all.

    Args:
      n: Local elements per core (before margin removal).
      m: Number of partitions along this axis.
      l: Local group size = ceil(n / m).
      axis: Which axis to shuffle (0, 1, or 2).
      margin_width: Width of halo/margin to skip.
    """
    position_index = lax.axis_index(self._axis_names[axis])
    raw_indices = jnp.arange(0, n)
    shift_indices = jnp.fmod(position_index * n, m) + raw_indices
    indices = (raw_indices // m) + jnp.fmod(shift_indices, m) * l
    return jnp.pad(
        jax.nn.one_hot(indices, num_classes=m * l),
        [[margin_width, margin_width], [0, 0]])

  def _get_gather_adjust_op(self, n, m, l, axis):
    """Build the post-all_to_all gather adjustment operator.

    After all_to_all, elements may not be in the correct local order.
    This operator re-orders them.
    """
    position_index = lax.axis_index(self._axis_names[axis])
    pos_indices = jnp.arange(0, m * l)
    source_position = pos_indices // l
    source_order = jnp.fmod(pos_indices, l)
    source_start = jnp.fmod(source_position * n, m)
    adjustment = jnp.where(
        jnp.less_equal(source_start, position_index), jnp.zeros([m * l]),
        m * jnp.ones([m * l]))
    mask = jnp.where(
        jnp.less(source_order * m + position_index - source_start + adjustment,
                 n), jnp.ones([m * l]), jnp.zeros([m * l]))
    upper_triangular = jnp.triu(jnp.ones([m * l, m * l]), 1)
    indices = jnp.int32(jnp.einsum('i,ij->j', mask, upper_triangular))
    indices = jnp.where(mask > 0, indices, -1 * jnp.ones([m * l]))
    return jax.nn.one_hot(indices, num_classes=n)

  def _get_correction_factor(self, n, m, axis, inverse):
    """Compute twiddle factors W_N^{pm} for phase correction.

    Args:
      n: Local size.
      m: Partition count along this axis.
      axis: Which axis (0, 1, 2).
      inverse: If True, compute inverse twiddle (conjugate + 1/m scaling).
    """
    inverse_factor = -1.0 if inverse else 1.0
    inverse_amplitude = 1.0 / m if inverse else 1.0
    position_index = lax.axis_index(self._axis_names[axis])
    twiddle_index_rem = (position_index * inverse_factor * jnp.arange(0, n)) % (
        n * m)
    factor = self._exp_itheta(
        (-2.0 * jnp.pi * twiddle_index_rem) / (n * m)) * inverse_amplitude
    for _ in range(2 - axis):
      factor = jnp.expand_dims(factor, -1)
    return factor

  # ─── Core Algorithm ─────────────────────────────────────────────────────

  def partitioned_fft_1d(self, partitioned_input, axis, inverse=False,
                         margin_width=0):
    """Run the 4-step distributed FFT pipeline on one axis.

    This is the CORE METHOD implementing the Cooley-Tukey M-way split:
      1. Shuffle + all_to_all  (data redistribution)
      2. Local FFT             (per-core computation)
      3. Twiddle multiply      (phase correction)
      4. Ring DFT via ppermute (cross-core DFT)

    Args:
      partitioned_input: Local 3D shard on this core.
      axis: Which axis to transform (0, 1, or 2).
      inverse: If True, compute the inverse FFT.
      margin_width: Halo width to exclude from transform.

    Returns:
      The transformed local shard.
    """
    input_shape = partitioned_input.shape
    if len(input_shape) != 3:
      raise ValueError('Unsupported input shape %s.' % str(input_shape))
    if axis not in [0, 1, 2]:
      raise ValueError('Unsupported axis: %d.' % axis)

    split_count = self._partition[axis]
    local_group_l = int(
        np.ceil((input_shape[axis] - 2 * margin_width) / split_count))
    n = (input_shape[axis] - 2 * margin_width)

    # ──── Step 1: Shuffle + All-to-All ────
    shuffle_op = self._get_shuffle_op(n, split_count, local_group_l, axis,
                                      margin_width)
    shuffled_input = jnp.einsum(
        self._shuffle_eqs[axis],
        partitioned_input,
        jnp.float32(shuffle_op),
        precision=lax.Precision.HIGHEST)

    # Avoid XLA padding on axis-2 when local size isn't 128-aligned
    all_to_all_dim = axis
    need_avoid_all_to_all_padding = (
        self._backend == 'tpu' and axis == 2 and (local_group_l % 128) != 0)
    if need_avoid_all_to_all_padding:
      shuffled_input = jnp.transpose(shuffled_input, [0, 2, 1])
      all_to_all_dim = 1

    gathered_input = lax.all_to_all(
        x=shuffled_input,
        axis_name=self._axis_names[axis],
        split_axis=all_to_all_dim,
        concat_axis=all_to_all_dim,
        tiled=True)

    if need_avoid_all_to_all_padding:
      gathered_input = jnp.transpose(gathered_input, [0, 2, 1])

    # Post-all_to_all reordering
    gather_adjust_op = self._get_gather_adjust_op(n, split_count,
                                                  local_group_l, axis)
    gathered_input = jnp.einsum(
        self._shuffle_eqs[axis],
        gathered_input,
        jnp.float32(gather_adjust_op),
        precision=lax.Precision.HIGHEST)

    # ──── Step 2: Local FFT ────
    gathered_input = jnp.transpose(gathered_input, self._fft_perms[axis])
    fft_input = jnp.complex64(gathered_input)
    local_fft = (
        jnp.fft.ifft(fft_input) if inverse else
        jnp.fft.fft(fft_input))
    local_fft = jnp.transpose(local_fft, self._fft_perms[axis])

    # ──── Step 3: Twiddle Multiply ────
    corr_factor = self._get_correction_factor(n, split_count, axis, inverse)
    adjusted_local_fft = local_fft * corr_factor

    # ──── Step 4: Ring DFT via ppermute ────
    LoopVars = collections.namedtuple('LoopVars', [
        'dest_fft', 'dest_core_position', 'source_fft',
        'source_core_position', 'i'
    ])

    def cond(v):
      return v.i < self._partition[axis]

    def body(v):
      inverse_factor = -1.0 if inverse else 1.0
      factor = self._exp_itheta(-2.0 * np.pi * inverse_factor *
                                v.dest_core_position *
                                v.source_core_position / split_count)
      dest_fft = v.dest_fft + factor * v.source_fft
      source_core_position = lax.ppermute(
          x=v.source_core_position,
          axis_name=self._axis_names[axis],
          perm=self._source_target_pairs[axis])
      source_fft = lax.ppermute(
          x=v.source_fft,
          axis_name=self._axis_names[axis],
          perm=self._source_target_pairs[axis])
      i = v.i + 1
      return LoopVars(dest_fft, dest_core_position, source_fft,
                      source_core_position, i)

    source_core_position = lax.axis_index(self._axis_names[axis])
    dest_core_position = source_core_position
    source_fft = adjusted_local_fft
    dest_fft = jnp.zeros_like(source_fft)

    i = 0
    dest_fft, _, _, _, _ = lax.while_loop(
        cond, body,
        init_val=LoopVars(dest_fft, dest_core_position, source_fft,
                          source_core_position, i))

    # Re-add margin padding
    padding = [[0, 0], [0, 0], [0, 0]]
    padding[axis] = [margin_width, margin_width]
    return jnp.pad(dest_fft, padding)

  # ─── High-Level API ─────────────────────────────────────────────────────

  def make_fft_3d_fn(self, global_shape, input_fn, kernel_fn):
    """Build a JIT-compiled shard_map function for the full 3D FFT pipeline.

    Returns a callable that runs: forward FFT → kernel multiply → inverse FFT
    entirely inside shard_map — data never leaves TPU.

    The output uses P('x', 'y', 'z') so shard_map automatically gathers
    all local shards and stitches them into the full global array.

    IMPORTANT: Call this ONCE to build the function, then call the returned
    function multiple times. The returned function is wrapped with @jax.jit,
    so the first call triggers XLA compilation and all subsequent calls
    reuse the cached compiled program.

    Args:
      global_shape: Shape of the full global array.
      input_fn: Function(local_shape, core_coord) → local input shard.
      kernel_fn: Function(local_shape, core_coord) → local kernel shard.

    Returns:
      A JIT-compiled callable: () → JAX array of shape global_shape.
    """
    partition_x, partition_y, partition_z = self._partition
    global_x, global_y, global_z = global_shape
    assert global_x % partition_x == 0
    assert global_y % partition_y == 0
    assert global_z % partition_z == 0
    nx = int(global_x / partition_x)
    ny = int(global_y / partition_y)
    nz = int(global_z / partition_z)
    self._create_mesh()

    # out_specs=P('x', 'y', 'z') tells shard_map that each core returns
    # a LOCAL shard, and shard_map should stitch them together into the
    # full GLOBAL array — like MPI_Gather assembling partial results.
    @jax.jit
    @functools.partial(
        shard_map, mesh=self._mesh,
        in_specs=(),
        out_specs=P(self._axis_names[0], self._axis_names[1],
                    self._axis_names[2]),
        check_rep=False)
    def do_transform():
      core_coord = (lax.axis_index(self._axis_names[0]),
                    lax.axis_index(self._axis_names[1]),
                    lax.axis_index(self._axis_names[2]))
      input_signal = input_fn((nx, ny, nz), core_coord)
      kernel = kernel_fn((nx, ny, nz), core_coord)
      # Forward 3D FFT (axis 0 → 1 → 2)
      forward_x = self.partitioned_fft_1d(input_signal, 0, False, 0)
      forward_xy = self.partitioned_fft_1d(forward_x, 1, False, 0)
      forward_xyz = self.partitioned_fft_1d(forward_xy, 2, False, 0)
      # Multiply in frequency domain
      forward_mul = forward_xyz * kernel
      # Inverse 3D FFT (axis 0 → 1 → 2)
      inverse_x = self.partitioned_fft_1d(forward_mul, 0, True, 0)
      inverse_xy = self.partitioned_fft_1d(inverse_x, 1, True, 0)
      out_signal = self.partitioned_fft_1d(inverse_xy, 2, True, 0)
      return out_signal

    return do_transform


print("✅ DistFFT class loaded successfully!")
print(f"   Source: https://github.com/google-research/swirl-lm/blob/main/swirl_lm/ext/fft/dist_fft.py")

# ═══════════════════════════════════════════════════════════════════════════════
# Workshop Configuration — auto-configured based on your hardware
# ═══════════════════════════════════════════════════════════════════════════════

# Enable highest precision for matrix operations
jax.config.update('jax_default_matmul_precision', 'highest')

# Create the DistFFT transformer and build the JIT-compiled function ONCE.
transformer = DistFFT(('x', 'y', 'z'), partition, backend=backend)
dist_fft_fn = transformer.make_fft_3d_fn(
    global_shape=GLOBAL_SHAPE, input_fn=input_fn, kernel_fn=kernel_fn)

print()
print("⚙️  Workshop Configuration")
print("=" * 50)
print(f"  Backend:         {backend}")
print(f"  TPU cores:       {num_cores}")
print(f"  Partition:       {partition}  ({partition[0]}×{partition[1]}×{partition[2]} = {np.prod(partition)})")
print(f"  Global shape:    {GLOBAL_SHAPE}")
print(f"  Local shard:     ({GLOBAL_SHAPE[0]//partition[0]}, {GLOBAL_SHAPE[1]//partition[1]}, {GLOBAL_SHAPE[2]//partition[2]})")
if backend != 'tpu':
    print(f"  Mode:            {backend.upper()} (single-device, no TPU acceleration)")
elif num_cores == 1:
    print(f"  Mode:            Single-device TPU (distributed ops are no-ops)")
else:
    print(f"  Mode:            Distributed across {num_cores} TPU cores")
print("=" * 50)

# ═══════════════════════════════════════════════════════════════════════════════
# Run the Distributed 3D FFT & Validate
# ═══════════════════════════════════════════════════════════════════════════════

print("\n🚀 Running Distributed 3D FFT")
print("=" * 50)

# Warmup — triggers XLA compilation (this is slow, ~30-60s)
_ = dist_fft_fn().block_until_ready()

# Run again — now reusing the cached compiled XLA program (fast!)
start = time.perf_counter()
for _ in range(NUM_TRIALS):
    jax_multicore_result = dist_fft_fn().block_until_ready()
jax_multicore_time = (time.perf_counter() - start) / NUM_TRIALS

print(f"\n⏱  3D FFT → kernel multiply → 3D iFFT ({M}×{M}×{M} = {M**3:,} elements):")
print(f"     NumPy (CPU):               {numpy_time*1000:.2f} ms")
print(f"     JAX (1-core {backend.upper()}):     {jax_time*1000:>10.2f} ms")
print(f"     DistFFT ({num_cores}-core {backend.upper()}): {jax_multicore_time*1000:>10.2f} ms")

if numpy_time > jax_multicore_time:
    print(f"     Speedup vs NumPy:          {numpy_time/jax_multicore_time:.1f}x 🚀")
if jax_time > jax_multicore_time:
    print(f"     Speedup vs 1-core:         {jax_time/jax_multicore_time:.1f}x 🚀")

# ═══════════════════════════════════════════════════════════════════════════════
# Validate Against NumPy Ground Truth
# ═══════════════════════════════════════════════════════════════════════════════
# dist_fft_fn() returns the FULL global array — shard_map automatically gathers
# all local shards from each core (via out_specs) and stitches them into the
# complete result.
#
# We report both L2 (overall quality) and L∞ (peak error) relative metrics
# to confirm the distributed algorithm is correct.

dist_rel_l2 = relative_l2_error(jax_multicore_result, numpy_result)
dist_rel_linf = relative_linfinite_error(jax_multicore_result, numpy_result)

print(f"\n✅ Correctness (DistFFT vs NumPy ground truth):")
print(f"     Relative L2 error:    {dist_rel_l2:.2e}   (overall quality)")
print(f"     Relative L∞ error:    {dist_rel_linf:.2e}   (peak error)")

print("\n🎉 The distributed FFT matches NumPy's result!")
print("     This confirms our TPU implementation is correct.")

if backend != 'tpu':
    print(f"\n📝 Running on {backend.upper()} — the algorithm and code are identical")
    print(f"     to the TPU version. On TPU, you'd see hardware-accelerated")
    print(f"     matmuls via the Matrix Multiply Unit (MXU) and, with multiple")
    print(f"     cores, real distributed communication across devices.")
elif num_cores == 1:
    print(f"\n📝 With 1 core, this is a standard single-device FFT.")
    print(f"     The algorithm structure (shuffle → local FFT → twiddle → ring DFT)")
    print(f"     is the same, but all_to_all and ppermute are no-ops.")
    print(f"     With 8 cores, the same code distributes work across all devices.")

✅ DistFFT class loaded successfully!
   Source: https://github.com/google-research/swirl-lm/blob/main/swirl_lm/ext/fft/dist_fft.py

⚙️  Workshop Configuration
  Backend:         tpu
  TPU cores:       8
  Partition:       (2, 2, 2)  (2×2×2 = 8)
  Global shape:    (256, 256, 256)
  Local shard:     (128, 128, 128)
  Mode:            Distributed across 8 TPU cores

🚀 Running Distributed 3D FFT

⏱  3D FFT → kernel multiply → 3D iFFT (256×256×256 = 16,777,216 elements):
     NumPy (CPU):               1215.56 ms
     JAX (1-core TPU):          34.89 ms
     DistFFT (8-core TPU):      17.56 ms
     Speedup vs NumPy:          69.2x 🚀
     Speedup vs 1-core:         2.0x 🚀

✅ Correctness (DistFFT vs NumPy ground truth):
     Relative L2 error:    5.25e-07   (overall quality)
     Relative L∞ error:    6.83e-07   (peak error)

🎉 The distributed FFT matches NumPy's result!
     This confirms our TPU implementation is correct.


<a id="section-4"></a>
## 🎯 Section 4: Key Takeaways

### What We Learned

| Concept | Key Takeaway | Familiar Analogy |
|---------|-------------|-----------------
| **TPU MXU** | 128×128 systolic array — makes matrix operations blazingly fast | Like a dedicated ASIC for General Matrix to Matrix multiplication (GeMM) |
| **TPU VPU** | Vector processing unit — handles element-wise ops (add, sin, cos, exp) | Like a Single Instruction/Multiple Data (SIMD) unit in a CPU |
| **TPU Mesh** | Logical organization of TPU cores into named axes | Like a Cartesian topology |
| **JAX** | Google's library for accelerator-optimized numerical computing | Like writing CUDA kernels, but with NumPy syntax |
| **`jax.lax`** | Low-level primitives for cross-device communication | Like Message Passing Interface (MPI) calls (`MPI_Alltoall`, `MPI_Sendrecv`) |
| **`shard_map`** | SPMD programming model — one function runs on all devices | Like MPI's SPMD model — same code, different data per rank |

### 📚 Further Reading

- [Cloud TPU Documentation](https://cloud.google.com/tpu/docs) — TPU architecture, setup, and best practices
- [JAX Documentation](https://jax.readthedocs.io/) — Official docs, tutorials, and API reference
- [XLA (Accelerated Linear Algebra)](https://www.tensorflow.org/xla) — The compiler backend that JAX uses to generate optimized code for TPU, GPU, and CPU

---

<table>
<tr>
<td>

**Resources:**
- 📦 [Swirl-LM GitHub](https://github.com/google-research/swirl-lm)
- 📄 [IEEE Paper](https://ieeexplore.ieee.org/document/9465154)
- 📖 [JAX Docs](https://jax.readthedocs.io/)

</td>
<td>

**Questions?**
Feel free to reach out or open an issue on the
[Swirl-LM GitHub repository](https://github.com/google-research/swirl-lm/issues).

</td>
</tr>
</table>

> *"The fastest code is the code that doesn't run — unless it runs on TPU."* 🚀